In [3]:
# Import the necessary libraries and load the data.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
df = pd.read_csv("/Users/serhatguldogan/Library/CloudStorage/OneDrive-KocUniversitesi/Project_/RAW DATA/BTC_1sec.csv")


We do the same thing that we did in "2_states_transitions_and_means.ipynb", but for 4 states. The price changes are -20,-10,10,20. 


We first classify the price changes as -2,-1,1,2 by a discretization trick.

In [4]:
# Calculate price changes
df['price_change'] = df['midpoint'].diff()

# Function to discretize price changes into two magnitude levels per sign:
# Positive bins: 0 < p ≤ 10 -> 10 (label 1)
#                p > 10      -> 20 (label 2)
# Negative bins are symmetric; 0 stays 0
def discretize_price_change(p):
    if pd.isna(p) or p == 0:
        return 0.0
    if p > 0:
        # small positive move -> 50, larger -> 100
        return 10.0 if p <= 10.0 else 20.0
    # p < 0: symmetric
    return -10.0 if p >= -10.0 else -20.0

# Apply discretization
df['price_change_discretized'] = df['price_change'].apply(discretize_price_change)

def label_price_change(discretized_value):
    if discretized_value == 0.0:
        return 0
    return int(discretized_value / 10)

df['price_change_label'] = df['price_change_discretized'].apply(label_price_change)

# Display results
print("First 20 rows with price changes:")
print(df[['midpoint', 'price_change', 'price_change_discretized', 'price_change_label']].head(20))
print("\nPrice change label distribution:")
print(df['price_change_label'].value_counts().sort_index())


First 20 rows with price changes:
     midpoint  price_change  price_change_discretized  price_change_label
0   56035.995           NaN                       0.0                   0
1   56035.995         0.000                       0.0                   0
2   56035.995         0.000                       0.0                   0
3   56035.995         0.000                       0.0                   0
4   56035.995         0.000                       0.0                   0
5   56035.995         0.000                       0.0                   0
6   56035.995         0.000                       0.0                   0
7   56035.995         0.000                       0.0                   0
8   56035.995         0.000                       0.0                   0
9   56035.995         0.000                       0.0                   0
10  56035.995         0.000                       0.0                   0
11  56035.995         0.000                       0.0                   0
12  

Keep track of the changes.

In [5]:
changes = df["price_change_label"]

In [ ]:
from collections import Counter

# Count the number of transitions between states in 'changes'
transition_counter = Counter()
for i in range(1, len(changes)):
    prev_state = changes.iloc[i - 1]
    curr_state = changes.iloc[i]
    
    transition_counter[(prev_state, curr_state)] += 1


How many times it moved from state $i \to j$. $i$ and $j$ specify the price movement, of course.

In [22]:
counter = dict(transition_counter)
counter

{(0, 0): 491297,
 (0, -2): 12741,
 (-2, -1): 10927,
 (-1, 1): 51381,
 (1, -1): 50233,
 (-1, -2): 10051,
 (-2, -2): 5619,
 (-2, 1): 6994,
 (-1, -1): 53339,
 (1, 1): 64847,
 (1, -2): 3776,
 (1, 0): 57212,
 (0, 2): 11766,
 (2, -2): 2299,
 (-1, 2): 3330,
 (2, 2): 5631,
 (2, 1): 11704,
 (-2, 2): 2550,
 (-2, 0): 8396,
 (2, -1): 5713,
 (1, 2): 9919,
 (2, 0): 7849,
 (0, 1): 51061,
 (0, -1): 44991,
 (-1, 0): 47101}

We again remove the movements that involves no price changes. For example, we remove $0 \to 2$, since we are interested in calculating the transition probabilities of nontrivial price movements. 

In [8]:
d=counter.copy()
for key in counter.keys():
    if 0 in key:
        d.pop(key)

Normalise it.

In [10]:
sum_1x = sum(v for (a, b), v in d.items() if a == 1)
normalized_1y = {(a, b): v / sum_1x for (a, b), v in d.items() if a == 1}
normalized_1y


{(1, -1): 0.3900834789361289,
 (1, 1): 0.5035682391768589,
 (1, -2): 0.0293224616579305,
 (1, 2): 0.07702582022908173}

In [11]:
normalized_by_first = {}
for a in set(key[0] for key in d.keys()):
    sum_ax = sum(v for (x, y), v in d.items() if x == a)
    if sum_ax == 0:
        continue
    for (x, y), v in d.items():
        if x == a:
            normalized_by_first[(x, y)] = v / sum_ax
normalized_by_first


{(1, -1): 0.3900834789361289,
 (1, 1): 0.5035682391768589,
 (1, -2): 0.0293224616579305,
 (1, 2): 0.07702582022908173,
 (2, -2): 0.09070106916005839,
 (2, 2): 0.22215646822109125,
 (2, 1): 0.46175089754211546,
 (2, -1): 0.2253915650767349,
 (-2, -1): 0.41881947106170947,
 (-2, -2): 0.21536987351475662,
 (-2, 1): 0.2680720582598697,
 (-2, 2): 0.09773859716366425,
 (-1, 1): 0.43505982167805524,
 (-1, -2): 0.08510512188719825,
 (-1, -1): 0.45163885149151994,
 (-1, 2): 0.028196204943226562}

This acts as a naive validation that the matrix will showcase valid transition probabilites.

In [12]:
t=0
for key in normalized_by_first.keys():
    if key[0]==2:
        t+=normalized_by_first[key]

t

1.0

Now get the transition matrix.

In [23]:
import numpy as np

# matrix will be ordered so rows/cols go from -4 to 4 skipping 0, i.e.
# indices: [-4, -3, -2, -1, 1, 2, 3, 4] (four magnitude levels per sign -> 8 non-zero states)
indices = [-2, -1, 1, 2]
size = len(indices)
mat = np.zeros((size, size))

# fill mat[i,j] with normalized_by_first[(row_val,col_val)]
for i, row in enumerate(indices):
    for j, col in enumerate(indices):
        key = (row, col)
        if key in normalized_by_first:
            mat[i, j] = normalized_by_first[key]
        else:
            mat[i, j] = 0.0  # or np.nan if preferred

mat  # this will display the matrix in the notebook


array([[0.21536987, 0.41881947, 0.26807206, 0.0977386 ],
       [0.08510512, 0.45163885, 0.43505982, 0.0281962 ],
       [0.02932246, 0.39008348, 0.50356824, 0.07702582],
       [0.09070107, 0.22539157, 0.4617509 , 0.22215647]])

Save the matrix for us to use it again.

In [14]:
np.savetxt("mat_4_states.csv",X=mat,fmt="%f",delimiter = ",")

Solve the eigenvalue problem to get the stationary probabilities of the Markov chain.

In [15]:

# Solve the equation vP = v where P is the matrix "mat" (i.e., find the stationary distribution)

# To solve vP = v, equivalently solve v(P - I) = 0 with constraint sum(v) = 1
# We transpose so that v is a row vector: v @ P = v <=> (P.T - I.T) v^T = 0
from numpy.linalg import svd

P = mat
size = P.shape[0]
A = P.T - np.eye(size)  # (P^T - I) so that nullspace is stationary v^T

# Append the normalization equation sum(v) = 1
A = np.vstack([A, np.ones(size)])
b = np.zeros(size + 1)
b[-1] = 1

# Solve using least squares (the last row enforces sum(v) = 1)
v, residuals, rank, s = np.linalg.lstsq(A, b, rcond=None)

# v is the stationary distribution
v = v.real  # ensure no tiny imaginary part from solver

v = np.array(v) # Display the stationary distribution vector


Save the stationary probability distribution vector.

In [16]:
# Save stationary distribution (use descriptive filename with extension)
np.savetxt(fname="pi_4_states.csv", X=v, fmt="%f",delimiter=",")

In [17]:
# Calculate mean price change from stationary distribution
# Each state j corresponds to price change = j * 25 (because bins are 25,50,75,100)
bucket_size = 10.0

# Calculate mean: E[X] = Σ (state_j * price_change_j * probability_j)
# where price_change_j = state_j * bucket_size
mean_price_change = 0.0

print("State | Price Change | Probability | Contribution")
print("-" * 60)
for i, state in enumerate(indices):
    price_change = state * bucket_size
    prob = v[i]
    contribution = price_change * prob
    mean_price_change += contribution
    if abs(contribution) > 0.001:  # Only print significant contributions
        print(f"{state:5d} | {price_change:11.2f} | {prob:10.6f} | {contribution:12.6f}")

print("-" * 60)
print(f"\nMean price change (from stationary distribution): {mean_price_change:.6f}")


State | Price Change | Probability | Contribution
------------------------------------------------------------
   -2 |      -20.00 |   0.069005 |    -1.380091
   -1 |      -10.00 |   0.405745 |    -4.057445
    1 |       10.00 |   0.456652 |     4.566524
    2 |       20.00 |   0.068598 |     1.371970
------------------------------------------------------------

Mean price change (from stationary distribution): 0.500957
